# Step 01 — Data Ingestion

This notebook explores the raw macro-economic time series scraped from multpl.com
and pulled from the FRED API. It visualizes coverage, checks for gaps, and provides
a quick sanity check on the ingested data before feature engineering.

**Prerequisites:** Run `python pipelines/01_ingest.py` first (or `python run_pipeline.py --steps 1`).

## Setup & Imports

In [ ]:
import sys
sys.path.insert(0, "../src")

import logging
import pathlib

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from market_regime.config import load, setup_logging
from market_regime.runtime import RunConfig
from market_regime import plotting, DATA_DIR

setup_logging()
log = logging.getLogger(__name__)

cfg = load()
run_cfg = RunConfig(generate_plots=True, save_plots=True, show_plots=True)

print("Config loaded:", cfg.get("data", {}))
print("RunConfig:", run_cfg)

## Load Raw Macro Data

The raw data is stored as a parquet file at `data/raw/macro_raw.parquet`.
Each row is a quarter, each column is one macro series.

In [ ]:
raw_path = DATA_DIR / "raw" / "macro_raw.parquet"

try:
    raw = pd.read_parquet(raw_path)
    print(f"Loaded raw data: {raw.shape[0]} quarters × {raw.shape[1]} series")
    print(f"Date range: {raw.index.min()} — {raw.index.max()}")
except FileNotFoundError:
    print(f"ERROR: {raw_path} not found.")
    print("Run `python pipelines/01_ingest.py` first to generate the raw data file.")
    raw = None

## Column Names

In [ ]:
if raw is not None:
    print(f"Total columns: {len(raw.columns)}")
    print("\nAll column names:")
    for i, col in enumerate(sorted(raw.columns)):
        print(f"  {i+1:3d}. {col}")

## Raw Series Coverage Heatmap

Dark cells indicate data is available for that quarter/series combination.
Many series only have data from the 1960s or 1970s onward.

In [ ]:
if raw is not None:
    plotting.plot_raw_series_coverage(raw, run_cfg)

## Raw Series Sample

Line charts for a selection of key economic series to visually verify
the ingestion is correct and the data looks reasonable.

In [ ]:
if raw is not None:
    sample_series = ["sp500", "us_infl", "10yr_ustreas", "fred_gdp", "div_yield"]
    # Filter to only columns that exist
    available = [s for s in sample_series if s in raw.columns]
    print(f"Plotting series: {available}")
    if available:
        plotting.plot_raw_series_sample(raw, available, run_cfg)
    else:
        print("None of the sample series found in raw data. Available columns:")
        print(list(raw.columns[:20]))

## Coverage Statistics

Percentage of non-NaN values per column, sorted descending.
Columns with low coverage may have limited utility in clustering.

In [ ]:
if raw is not None:
    coverage_pct = raw.notna().mean().sort_values(ascending=False) * 100
    print("Coverage (% non-NaN) per column:")
    print("-" * 40)
    for col, pct in coverage_pct.items():
        bar = "#" * int(pct / 5)
        print(f"  {col:<35s} {pct:6.1f}%  {bar}")

## First and Last 5 Rows

In [ ]:
if raw is not None:
    print("=== FIRST 5 ROWS ===")
    display(raw.head())
    print("\n=== LAST 5 ROWS ===")
    display(raw.tail())

## Summary Statistics

In [ ]:
if raw is not None:
    print("Summary statistics (transposed for readability):")
    display(raw.describe().T.sort_values("count", ascending=False))